In [1]:
!git clone https://github.com/MohamedElsayed75/FRW1NB1.git
%cd FRW1NB1/work/notebooks

Cloning into 'FRW1NB1'...
remote: Enumerating objects: 134, done.
remote: Counting objects: 100% (134/134), done.
remote: Compressing objects: 100% (105/105), done.
remote: Total 134 (delta 47), reused 79 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (134/134), 1.89 MiB | 10.10 MiB/s, done.
Resolving deltas: 100% (47/47), done.
/content/FRW1NB1/work/notebooks


In [2]:
import os
import duckdb
import numpy as np
import pandas as pd

from google.colab import userdata

print("Imports ready.")

Imports ready.


In [3]:
HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError("HF_TOKEN was not found. Add it in Colab → Secrets.")

con = duckdb.connect()

con.execute(f"""
INSTALL httpfs;
LOAD httpfs;

CREATE OR REPLACE SECRET hf (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
);
""")

TABLE = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet"

print("Warehouse connection ready.")

Warehouse connection ready.


# ML-10 — Content Action Playbook

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [4]:
monthly = con.execute(f"""
SELECT
    client_hash_id,
    content_hash_id,
    month,

    AVG(gsc_clicks) AS avg_gsc_clicks,
    AVG(gsc_impressions) AS avg_gsc_impressions,
    AVG(gsc_avg_position) AS avg_position,
    AVG(ga4_pageviews) AS avg_ga4_pageviews,
    AVG(ga4_sessions) AS avg_ga4_sessions

FROM read_parquet('{TABLE}')

WHERE month IN ('2026-02', '2026-03')

GROUP BY
    client_hash_id,
    content_hash_id,
    month
""").fetchdf()

print("Rows:", len(monthly))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 652983


In [5]:
pivot = monthly.pivot_table(
    index=["client_hash_id", "content_hash_id"],
    columns="month",
    values=[
        "avg_gsc_clicks",
        "avg_gsc_impressions",
        "avg_position",
        "avg_ga4_pageviews",
        "avg_ga4_sessions"
    ],
    aggfunc="first"
)

pivot.columns = [
    f"{metric}_{month}"
    for metric, month in pivot.columns
]

queue = pivot.reset_index()

print("Queue rows:", len(queue))

Queue rows: 349411


In [6]:
queue["click_change_pct"] = np.where(
    queue["avg_gsc_clicks_2026-02"] > 0,
    (
        queue["avg_gsc_clicks_2026-03"]
        - queue["avg_gsc_clicks_2026-02"]
    )
    / queue["avg_gsc_clicks_2026-02"]
    * 100,
    np.nan
)

queue["impression_change_pct"] = np.where(
    queue["avg_gsc_impressions_2026-02"] > 0,
    (
        queue["avg_gsc_impressions_2026-03"]
        - queue["avg_gsc_impressions_2026-02"]
    )
    / queue["avg_gsc_impressions_2026-02"]
    * 100,
    np.nan
)

In [7]:
queue["action_score"] = 0

queue.loc[
    queue["click_change_pct"] <= -20,
    "action_score"
] += 40

queue.loc[
    (queue["click_change_pct"] > -20) &
    (queue["click_change_pct"] < 0),
    "action_score"
] += 20

queue.loc[
    queue["impression_change_pct"] <= -20,
    "action_score"
] += 40

queue.loc[
    (queue["impression_change_pct"] > -20) &
    (queue["impression_change_pct"] < 0),
    "action_score"
] += 20

queue.loc[
    queue["avg_gsc_impressions_2026-03"] >= 100,
    "action_score"
] += 20

In [8]:
def get_reason_code(row):

    click_down = (
        pd.notna(row["click_change_pct"]) and
        row["click_change_pct"] < 0
    )

    impressions_down = (
        pd.notna(row["impression_change_pct"]) and
        row["impression_change_pct"] < 0
    )

    if click_down and impressions_down:
        return "DOUBLE_DECLINE"

    if click_down:
        return "CLICK_DECLINE"

    if impressions_down:
        return "IMPRESSION_DECLINE"

    if row["avg_gsc_impressions_2026-03"] >= 100:
        return "HIGH_VISIBILITY"

    return "NO_STRONG_SIGNAL"


queue["reason_code"] = queue.apply(
    get_reason_code,
    axis=1
)

In [9]:
def get_action(row):

    score = row["action_score"]

    if score >= 60:
        return "REVIEW_REFRESH"

    if score >= 40:
        return "REVIEW"

    if score >= 20:
        return "MONITOR"

    return "NO_ACTION"


queue["action"] = queue.apply(
    get_action,
    axis=1
)

In [10]:
queue = queue.sort_values(
    ["action_score", "reason_code"],
    ascending=[False, True]
).reset_index(drop=True)

queue["rank"] = np.arange(1, len(queue) + 1)

queue[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "action_score",
        "reason_code",
        "action",
        "click_change_pct",
        "impression_change_pct"
    ]
].head(20)

,rank,client_hash_id,content_hash_id,action_score,reason_code,action,click_change_pct,impression_change_pct
0,1,client_08a6a72ff48e62c0,content_02cf865ff739233b,100,DOUBLE_DECLINE,REVIEW_REFRESH,-66.881720,-47.281548
1,2,client_08a6a72ff48e62c0,content_0467482c82a8a797,100,DOUBLE_DECLINE,REVIEW_REFRESH,-60.860215,-38.690270
2,3,client_08a6a72ff48e62c0,content_05fd36dfc57842c7,100,DOUBLE_DECLINE,REVIEW_REFRESH,-72.208437,-55.145890
3,4,client_08a6a72ff48e62c0,content_06161601bd898c97,100,DOUBLE_DECLINE,REVIEW_REFRESH,-55.940205,-30.819498
4,5,client_08a6a72ff48e62c0,content_066f6f52517a70ba,100,DOUBLE_DECLINE,REVIEW_REFRESH,-62.162162,-42.409231
5,6,client_08a6a72ff48e62c0,content_0704158fde70da26,100,DOUBLE_DECLINE,REVIEW_REFRESH,-58.709677,-52.203214
6,7,client_08a6a72ff48e62c0,content_07c2d3931ab3a789,100,DOUBLE_DECLINE,REVIEW_REFRESH,-22.580645,-23.721798
7,8,client_08a6a72ff48e62c0,content_09c96ca5175f1c37,100,DOUBLE_DECLINE,REVIEW_REFRESH,-22.580645,-34.745801
8,9,client_08a6a72ff48e62c0,content_0c88afbc33f48acb,100,DOUBLE_DECLINE,REVIEW_REFRESH,-22.580645,-21.417230
9,10,client_08a6a72ff48e62c0,content_0eeee44c675bca8e,100,DOUBLE_DECLINE,REVIEW_REFRESH,-57.215620,-20.755128


## 1. Ranked actions + reason codes

The queue ranks content using a simple rule-based score built from observed search and analytics signals.

Each row receives:

- an action score,
- one primary reason code,
- an action label,
- and a rank.

The queue is intended to prioritize human attention rather than automatically change content.

The highest-priority action is `REVIEW_REFRESH`, followed by `REVIEW`, `MONITOR`, and `NO_ACTION`.

Reason codes make the ranking auditable: a reviewer can see which observed signal caused an item to enter the queue.

In [11]:
archetype_actions = pd.DataFrame({
    "archetype": [
        "DOUBLE_DECLINE",
        "CLICK_DECLINE",
        "IMPRESSION_DECLINE",
        "HIGH_VISIBILITY",
        "NO_STRONG_SIGNAL"
    ],
    "recommended_action": [
        "Review for refresh; verify demand, SERP, and content relevance first.",
        "Review title/snippet, SERP position, and search intent before editing.",
        "Review ranking, search demand, indexing, and technical visibility.",
        "Prioritize for monitoring because the page has meaningful visibility.",
        "Do not prioritize without another supporting signal."
    ]
})

archetype_actions

,archetype,recommended_action
0,DOUBLE_DECLINE,"Review for refresh; verify demand, SERP, and c..."
1,CLICK_DECLINE,"Review title/snippet, SERP position, and searc..."
2,IMPRESSION_DECLINE,"Review ranking, search demand, indexing, and t..."
3,HIGH_VISIBILITY,Prioritize for monitoring because the page has...
4,NO_STRONG_SIGNAL,Do not prioritize without another supporting s...


### Archetype → action mapping

The mapping converts the numerical signals into practical review actions.

The mapping is deliberately advisory. A reason code identifies why an item was ranked, but it does not prove that the content requires an edit.

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

## 2. Intended use and limits

### Intended use

The playbook is intended to help a content team prioritize which pages deserve human investigation first.

The queue can be used as a decision-support tool for:

- prioritizing content review,
- identifying possible refresh candidates,
- identifying pages with declining search visibility,
- and organizing a review backlog by observable signals.

The output is directional. It does not establish that a page needs a refresh or that a refresh will improve performance.

### Limits

The current analysis is based on observed warehouse signals and a development target.

The Week-6 audit identified an important temporal limitation: the current development setup uses March observations both in the feature set and in the decline target. Therefore, this queue should not be interpreted as a proven future forecasting system.

The model and rule also do not observe important contextual factors such as editorial quality, search intent changes, competitor behaviour, seasonality, or business priorities.

The queue therefore supports human prioritization rather than automatic content decisions.

### Decay / refresh insight

The research paper reports that the 31–90 day freshness window had the strongest stable growth-to-decline ratio, while older content showed evidence that refreshing some aging pages can be associated with improved search health. The paper also cautions that very old-content buckets can be unstable when sample sizes are small.

This supports using content age and freshness as useful review context rather than as an automatic refresh trigger.

The practical implication for this playbook is that aging content with negative performance signals can be prioritized for human review, while freshness alone should not determine the action.

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

## 3. Human review + the no-go list

### Human review rules

Before taking any content action, a reviewer should:

1. Verify that the observed decline is not caused by a tracking or data-availability issue.
2. Check whether search demand or seasonality explains the change.
3. Inspect the current SERP position and search intent.
4. Check whether the page is still relevant to the target query.
5. Review the actual content before deciding whether a refresh is appropriate.
6. Confirm that the proposed change has a clear purpose.
7. Record the final human decision and, where possible, the reason for accepting or rejecting the recommendation.

### What should NOT be automated

The system should not automatically:

- publish content changes,
- delete or redirect pages,
- rewrite titles or content without review,
- change search intent,
- change canonical URLs,
- make technical SEO changes,
- declare a page successful or unsuccessful based only on the score,
- or claim that a refresh caused an improvement without an appropriate follow-up evaluation.

The score is a prioritization signal, not an autonomous content-management decision.

### Cost / value thinking

The queue should prioritize actions where the potential value of investigation is reasonably high relative to the cost of review.

High-visibility pages with meaningful declines may deserve earlier review because changes could affect a larger amount of observed search traffic.

Lower-visibility pages may still be worth reviewing when the cost is small or when they have strategic importance.

The score itself is not a financial value estimate. A human reviewer should consider traffic opportunity, business importance, effort required, and confidence in the underlying signal before deciding whether to act.

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

## 4. Monitoring / retrain triggers

The playbook should be monitored rather than treated as a permanent rule.

### Monitoring triggers

Review the system when:

- the distribution of action scores changes substantially,
- the proportion of `REVIEW_REFRESH` items changes unexpectedly,
- data availability or warehouse schemas change,
- GSC or GA4 coverage changes,
- the meaning of an input signal changes,
- or the observed relationship between the queue and subsequent content outcomes changes.

### Retrain / recalibration triggers

A model should be reconsidered when:

- new labelled outcome data becomes available,
- validation performance degrades materially,
- grouped or time-aware validation produces substantially different results,
- feature distributions drift,
- or the current rule no longer reflects the team's decision process.

Any retraining should use a clearly defined outcome window and a validation design that prevents future information from entering the features.

Monitoring should focus on whether the system remains useful for human decision support, not only whether a single metric remains high.

In [12]:
monitoring_triggers = pd.DataFrame({
    "trigger": [
        "Score distribution shift",
        "Unexpected action-volume change",
        "Data availability change",
        "Feature definition change",
        "Validation performance decline",
        "Feature distribution drift",
        "New labelled outcomes"
    ],
    "response": [
        "Investigate the cause before using the queue.",
        "Audit thresholds and input distributions.",
        "Pause affected recommendations and investigate.",
        "Revalidate the rule/model before continued use.",
        "Re-evaluate model versus baseline.",
        "Inspect drift and consider recalibration.",
        "Run a fresh validation experiment."
    ]
})

monitoring_triggers

,trigger,response
0,Score distribution shift,Investigate the cause before using the queue.
1,Unexpected action-volume change,Audit thresholds and input distributions.
2,Data availability change,Pause affected recommendations and investigate.
3,Feature definition change,Revalidate the rule/model before continued use.
4,Validation performance decline,Re-evaluate model versus baseline.
5,Feature distribution drift,Inspect drift and consider recalibration.
6,New labelled outcomes,Run a fresh validation experiment.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [13]:
output_dir = "../../work/outputs"

os.makedirs(
    output_dir,
    exist_ok=True
)

export_columns = [
    "rank",
    "client_hash_id",
    "content_hash_id",
    "action_score",
    "reason_code",
    "action",
    "click_change_pct",
    "impression_change_pct"
]

final_queue = queue[
    export_columns
].copy()

output_path = os.path.join(
    output_dir,
    "w07_ranked_action_queue.csv"
)

final_queue.to_csv(
    output_path,
    index=False
)

print("Saved:", output_path)
print("Rows:", len(final_queue))

Saved: ../../work/outputs/w07_ranked_action_queue.csv
Rows: 349411


In [14]:
archetype_path = os.path.join(
    output_dir,
    "w07_archetype_action_mapping.csv"
)

archetype_actions.to_csv(
    archetype_path,
    index=False
)

print("Saved:", archetype_path)

Saved: ../../work/outputs/w07_archetype_action_mapping.csv


In [15]:
import json

summary = {
    "queue_rows": int(len(final_queue)),
    "review_refresh_count": int(
        (final_queue["action"] == "REVIEW_REFRESH").sum()
    ),
    "review_count": int(
        (final_queue["action"] == "REVIEW").sum()
    ),
    "monitor_count": int(
        (final_queue["action"] == "MONITOR").sum()
    ),
    "no_action_count": int(
        (final_queue["action"] == "NO_ACTION").sum()
    ),
    "reason_code_counts": {
        str(k): int(v)
        for k, v in final_queue["reason_code"]
        .value_counts()
        .items()
    },
    "methodological_note": (
        "Development decision-support queue. "
        "Not a production forecasting system. "
        "Week-6 audit identified temporal leakage in the "
        "current development target/feature setup."
    )
}

json_path = os.path.join(
    output_dir,
    "w07_action_playbook_summary.json"
)

with open(json_path, "w") as f:
    json.dump(
        summary,
        f,
        indent=2
    )

print("Saved:", json_path)

Saved: ../../work/outputs/w07_action_playbook_summary.json


In [16]:
print("FINAL EXPORTED QUEUE")
print("====================")

display(
    final_queue.head(20)
)

FINAL EXPORTED QUEUE


,rank,client_hash_id,content_hash_id,action_score,reason_code,action,click_change_pct,impression_change_pct
0,1,client_08a6a72ff48e62c0,content_02cf865ff739233b,100,DOUBLE_DECLINE,REVIEW_REFRESH,-66.881720,-47.281548
1,2,client_08a6a72ff48e62c0,content_0467482c82a8a797,100,DOUBLE_DECLINE,REVIEW_REFRESH,-60.860215,-38.690270
2,3,client_08a6a72ff48e62c0,content_05fd36dfc57842c7,100,DOUBLE_DECLINE,REVIEW_REFRESH,-72.208437,-55.145890
3,4,client_08a6a72ff48e62c0,content_06161601bd898c97,100,DOUBLE_DECLINE,REVIEW_REFRESH,-55.940205,-30.819498
4,5,client_08a6a72ff48e62c0,content_066f6f52517a70ba,100,DOUBLE_DECLINE,REVIEW_REFRESH,-62.162162,-42.409231
5,6,client_08a6a72ff48e62c0,content_0704158fde70da26,100,DOUBLE_DECLINE,REVIEW_REFRESH,-58.709677,-52.203214
6,7,client_08a6a72ff48e62c0,content_07c2d3931ab3a789,100,DOUBLE_DECLINE,REVIEW_REFRESH,-22.580645,-23.721798
7,8,client_08a6a72ff48e62c0,content_09c96ca5175f1c37,100,DOUBLE_DECLINE,REVIEW_REFRESH,-22.580645,-34.745801
8,9,client_08a6a72ff48e62c0,content_0c88afbc33f48acb,100,DOUBLE_DECLINE,REVIEW_REFRESH,-22.580645,-21.417230
9,10,client_08a6a72ff48e62c0,content_0eeee44c675bca8e,100,DOUBLE_DECLINE,REVIEW_REFRESH,-57.215620,-20.755128


In [17]:
print("Action counts:")
print(
    final_queue["action"].value_counts()
)

print("\nReason-code counts:")
print(
    final_queue["reason_code"].value_counts()
)

Action counts:
action
NO_ACTION         251501
REVIEW             53575
MONITOR            23348
REVIEW_REFRESH     20987
Name: count, dtype: int64

Reason-code counts:
reason_code
NO_STRONG_SIGNAL      251501
IMPRESSION_DECLINE     52174
DOUBLE_DECLINE         18288
CLICK_DECLINE          16549
HIGH_VISIBILITY        10899
Name: count, dtype: int64


### Exports for the paper

The notebook exports the ranked action queue and archetype-to-action mapping into `work/outputs/`.

The queue provides the concrete action layer for the recommendations section of the research paper.

The exported results should be described as a research/development decision-support artifact rather than a production deployment.

The paper should report the methodology, limitations, and human-review requirements alongside the recommendations.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

In [18]:
print("SELF-CHECK")
print("===========")

checks = {
    "Ranked action queue created": len(final_queue) > 0,
    "Reason codes included": "reason_code" in final_queue.columns,
    "Action labels included": "action" in final_queue.columns,
    "Intended use documented": True,
    "Limits documented": True,
    "Human review rules documented": True,
    "No-go list documented": True,
    "Monitoring triggers documented": True,
    "Cost/value thinking documented": True,
    "Queue exported": os.path.exists(output_path),
    "Archetype mapping exported": os.path.exists(archetype_path),
    "Summary JSON exported": os.path.exists(json_path),
    "Automation limits stated": True,
    "Week-6 leakage limitation acknowledged": True
}

for name, passed in checks.items():
    print(
        f"[{'PASS' if passed else 'FAIL'}] {name}"
    )

SELF-CHECK
[PASS] Ranked action queue created
[PASS] Reason codes included
[PASS] Action labels included
[PASS] Intended use documented
[PASS] Limits documented
[PASS] Human review rules documented
[PASS] No-go list documented
[PASS] Monitoring triggers documented
[PASS] Cost/value thinking documented
[PASS] Queue exported
[PASS] Archetype mapping exported
[PASS] Summary JSON exported
[PASS] Automation limits stated
[PASS] Week-6 leakage limitation acknowledged
